In [ ]:
from src.container import ServiceContainer
from src.utils.app_utils import load_env_vars
from src.utils.log_utils import silence_loggers
from tqdm import tqdm
from src.stages import Stage
import json
import os

In [ ]:
load_env_vars()
silence_loggers()

In [ ]:
container = ServiceContainer.create_real()

# Find all broken snaps

In [ ]:
blobs = container.storage.list_blobs_nest()

In [ ]:
broken_keys = []
snaps = blobs[Stage.SNAP.value]

for company in tqdm(snaps, position=0):
    for policy in tqdm(snaps[company], position=1, leave=False):
        for ts in snaps[company][policy]:
            parts = (Stage.SNAP.value, company, policy, ts)
            path = container.storage.unparse_blob_path(parts)
            key = path.removeprefix(f"{Stage.SNAP.value}/").removesuffix(".html")
            txt = container.storage.load_text_blob(path)
            if "ask the publisher" in txt.lower():
                broken_keys.append(key)

In [ ]:
len(broken_keys)

# Curate snapshot manifests

In [ ]:
for key in broken_keys:
    company, policy, ts = key.split('/')
    path = '/'.join((Stage.META.value, company, policy, "manifest.json"))
    manifest = container.storage.load_json_blob(path)
    manifest_rows = [row for row in manifest if row['timestamp'] != ts]
    container.storage.upload_json_blob(json.dumps(manifest_rows), path)


In [ ]:
for key in broken_keys:
    company, policy, ts = key.split('/')
    path = '/'.join((Stage.META.value, company, policy, "metadata.json"))
    manifest = container.storage.load_json_blob(path)
    manifest_rows = [row for row in manifest if row[1] != ts]
    container.storage.upload_json_blob(json.dumps(manifest_rows), path)


# Curate Diff Manifests

In [ ]:
more_broken_keys = []
for key in broken_keys:
    company, policy, ts = key.split('/')
    name = f"{ts}.json"
    path = '/'.join((Stage.DIFF_RAW.value, company, policy, "manifest.json"))
    manifest = container.storage.load_json_blob(path)
    manifest_rows = {}
    for before, after in manifest.items():
        if before == name or after == name:
            more_broken_keys.append('/'.join((company, policy, before)).removesuffix(".json"))
            more_broken_keys.append('/'.join((company, policy, after)).removesuffix(".json"))
        else:
            manifest_rows[before] = after
    container.storage.upload_json_blob(json.dumps(manifest_rows), path)


In [ ]:
len(more_broken_keys)

In [ ]:
broken_keys += more_broken_keys

# Delete downstream artifacts

In [ ]:
blobs_flat = container.storage.adapter.list_blobs()

In [ ]:
for key in broken_keys:
    for stage in Stage:
        path = '/'.join((stage.value, key))
        for blob in blobs_flat:
            if blob.startswith(path):
                container.storage.remove_blob(path)

# Delete Labels

In [ ]:
for version in os.listdir("../../data/eval"):
    label_file = f"../../data/eval/{version}/records.json"
    with open(label_file) as f:
        data = json.load(f)
    fixed_data = []
    for row in data:
        if not any(key in row['metadata']['blob_path'] for key in broken_keys):
            fixed_data.append(row)
    print(f"Writing back {len(fixed_data)} of {len(data)} records found in {version}")
    with open(label_file, "w") as f:
        json.dump(fixed_data, f)